In [ ]:
# Earthquake Data Pipeline (USGS API)

This notebook validates, transforms, and analyzes earthquake data extracted from the USGS public API.

In [ ]:
## 1. Environment Setup

In [7]:
import os
os.getcwd()

'c:\\Users\\jarci\\New folder\\basic_pipeline\\notebooks'

In [9]:
import os
os.listdir("../data/raw")

['earthquakes_last_15_days.csv']

In [ ]:
import pandas as pd
df = pd.read_csv("../data/raw/earthquakes_last_15_days.csv")
df.shape

(4183, 10)

In [ ]:
## 2. Data Validation & Quality Checks

In [12]:
df.head(3)

,id,time_ms,place,mag,type,url,longitude,latitude,depth_km,time_utc
0,nc75295851,1768434574240,"8 km NW of The Geysers, CA",0.76,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,-122.819168,38.832333,1.77,2026-01-14 23:49:34.240000+00:00
1,av93994861,1768434297920,"5 km W of Dutch Harbor, Alaska",-0.44,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,-166.625333,53.894333,7.83,2026-01-14 23:44:57.920000+00:00
2,ci41158007,1768434142140,"16 km WSW of Johannesburg, CA",1.35,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,-117.810167,35.344000,6.95,2026-01-14 23:42:22.140000+00:00


In [13]:
df.columns 

Index(['id', 'time_ms', 'place', 'mag', 'type', 'url', 'longitude', 'latitude',
       'depth_km', 'time_utc'],
      dtype='object')

In [14]:
df.dtypes

id            object
time_ms        int64
place         object
mag          float64
type          object
url           object
longitude    float64
latitude     float64
depth_km     float64
time_utc      object
dtype: object

In [15]:
df.isna().sum().sort_values(ascending=False) 

mag          1
id           0
time_ms      0
place        0
type         0
url          0
longitude    0
latitude     0
depth_km     0
time_utc     0
dtype: int64

In [16]:
df.duplicated().sum()

0

In [17]:
df["id"].duplicated().sum()

0

In [18]:
df["mag"].describe()

count    4182.000000
mean        1.643715
std         1.243625
min        -1.440000
25%         0.850000
50%         1.510000
75%         2.100000
max         6.500000
Name: mag, dtype: float64

In [19]:
df[["latitude","longitude","depth_km"]].describe()

,latitude,longitude,depth_km
count,4183.000000,4183.000000,4183.000000
mean,40.440017,-113.367255,20.059932
std,17.727327,63.189269,44.849307
min,-59.920200,-179.999200,-3.290000
25%,32.881000,-150.253000,3.060000
50%,38.815333,-122.777000,7.240000
75%,58.258500,-112.369833,14.584400
max,70.818700,179.773600,619.726000


In [20]:
out_lat = df[(df["latitude"] < -90) | (df["latitude"] > 90)]
out_lon = df[(df["longitude"] < -180) | (df["longitude"] > 180)]
len(out_lat), len(out_lon)


(0, 0)

In [21]:
df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce")
df["time_utc"].min(), df["time_utc"].max()


(Timestamp('2025-12-31 00:03:35.983000+0000', tz='UTC'),
 Timestamp('2026-01-14 23:49:34.240000+0000', tz='UTC'))

In [22]:
df["time_utc"].isna().sum()


24

In [23]:
quality_report = {
    "rows": len(df),
    "cols": df.shape[1],
    "missing_total": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_ids": int(df["id"].duplicated().sum()),
    "mag_missing": int(df["mag"].isna().sum()),
    "time_utc_missing": int(df["time_utc"].isna().sum()),
}

quality_report


{'rows': 4183,
 'cols': 10,
 'missing_total': 25,
 'duplicate_rows': 0,
 'duplicate_ids': 0,
 'mag_missing': 1,
 'time_utc_missing': 24}

In [24]:
import pandas as pd

df_raw = pd.read_csv("../data/raw/earthquakes_last_15_days.csv")
df_raw.shape

(4183, 10)

In [25]:
df = df_raw.copy() 
df.shape

(4183, 10)

In [26]:
df.isna().sum() 

id           0
time_ms      0
place        0
mag          1
type         0
url          0
longitude    0
latitude     0
depth_km     0
time_utc     0
dtype: int64

In [27]:
df = df[df["mag"].notna()]
df.shape

(4182, 10)

In [ ]:
## 3. Data Transformation

In [32]:
#convertir a datetime 
df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce")


In [33]:
df = df[df["time_utc"].notna()]
df.shape

#eliminar filas sin fecha valida


(4158, 10)

In [34]:
df.columns

Index(['id', 'time_ms', 'place', 'mag', 'type', 'url', 'longitude', 'latitude',
       'depth_km', 'time_utc'],
      dtype='object')

In [35]:
columns_to_keep = [
    "id",
    "time_utc",
    "place",
    "mag",
    "latitude",
    "longitude",
    "depth_km",
    "type"
]

df = df[columns_to_keep]
df.shape


(4158, 8)

In [36]:
df["event_hour"] = df["time_utc"].dt.hour
df.head(3)

,id,time_utc,place,mag,latitude,longitude,depth_km,type,event_hour
0,nc75295851,2026-01-14 23:49:34.240000+00:00,"8 km NW of The Geysers, CA",0.76,38.832333,-122.819168,1.77,earthquake,23
1,av93994861,2026-01-14 23:44:57.920000+00:00,"5 km W of Dutch Harbor, Alaska",-0.44,53.894333,-166.625333,7.83,earthquake,23
2,ci41158007,2026-01-14 23:42:22.140000+00:00,"16 km WSW of Johannesburg, CA",1.35,35.344000,-117.810167,6.95,earthquake,23


In [37]:
df["event_date"] = df["time_utc"].dt.date
df.head(3)

,id,time_utc,place,mag,latitude,longitude,depth_km,type,event_hour,event_date
0,nc75295851,2026-01-14 23:49:34.240000+00:00,"8 km NW of The Geysers, CA",0.76,38.832333,-122.819168,1.77,earthquake,23,2026-01-14
1,av93994861,2026-01-14 23:44:57.920000+00:00,"5 km W of Dutch Harbor, Alaska",-0.44,53.894333,-166.625333,7.83,earthquake,23,2026-01-14
2,ci41158007,2026-01-14 23:42:22.140000+00:00,"16 km WSW of Johannesburg, CA",1.35,35.344000,-117.810167,6.95,earthquake,23,2026-01-14


In [38]:
df.shape
df.columns

Index(['id', 'time_utc', 'place', 'mag', 'latitude', 'longitude', 'depth_km',
       'type', 'event_hour', 'event_date'],
      dtype='object')

In [39]:
df.shape

(4158, 10)

In [40]:
df.isna().sum()

id            0
time_utc      0
place         0
mag           0
latitude      0
longitude     0
depth_km      0
type          0
event_hour    0
event_date    0
dtype: int64

In [42]:
df.dtypes

id                         object
time_utc      datetime64[ns, UTC]
place                      object
mag                       float64
latitude                  float64
longitude                 float64
depth_km                  float64
type                       object
event_hour                  int32
event_date                 object
dtype: object

In [ ]:
df["event_hour"].min(), df["event_hour"].max()

(0, 23)

In [ ]:
## 4. Exploratory Data Analysis (EDA)

In [44]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/earthquakes_processed.csv", index=False)


In [45]:
import pandas as pd 

df = pd.read_csv("../data/processed/earthquakes_processed.csv")
df.shape

(4158, 10)

In [46]:
df.describe()

,mag,latitude,longitude,depth_km,event_hour
count,4158.000000,4158.000000,4158.000000,4158.000000,4158.000000
mean,1.644658,40.453468,-113.305973,20.114360,11.649591
std,1.243886,17.754591,63.352246,44.944958,6.944373
min,-1.440000,-59.920200,-179.999200,-3.290000,0.000000
25%,0.860000,32.870500,-150.317000,3.080000,6.000000
50%,1.510000,38.815166,-122.776085,7.250500,12.000000
75%,2.100000,58.264500,-112.051500,14.600000,18.000000
max,6.500000,70.818700,179.773600,619.726000,23.000000


In [47]:
df["mag"].describe()

count    4158.000000
mean        1.644658
std         1.243886
min        -1.440000
25%         0.860000
50%         1.510000
75%         2.100000
max         6.500000
Name: mag, dtype: float64

In [48]:
df["mag"].value_counts(bins=10).sort_index()

mag
(-1.4489999999999998, -0.646]     110
(-0.646, 0.148]                   179
(0.148, 0.942]                    868
(0.942, 1.736]                   1323
(1.736, 2.53]                     998
(2.53, 3.324]                     306
(3.324, 4.118]                     76
(4.118, 4.912]                    222
(4.912, 5.706]                     69
(5.706, 6.5]                        7
Name: count, dtype: int64

In [50]:
df["event_date"].value_counts().sort_index()

event_date
2025-12-31    311
2026-01-01    319
2026-01-02    272
2026-01-03    300
2026-01-04    296
2026-01-05    274
2026-01-06    331
2026-01-07    281
2026-01-08    266
2026-01-09    225
2026-01-10    222
2026-01-11    243
2026-01-12    229
2026-01-13    313
2026-01-14    276
Name: count, dtype: int64

In [51]:
df["event_hour"].value_counts().sort_index()

event_hour
0     182
1     169
2     146
3     157
4     165
5     162
6     196
7     177
8     187
9     168
10    185
11    176
12    181
13    156
14    157
15    182
16    168
17    148
18    180
19    161
20    208
21    176
22    192
23    179
Name: count, dtype: int64

In [52]:
df["place"].value_counts().head(10)


place
8 km NW of The Geysers, CA        64
7 km NW of The Geysers, CA        45
6 km NW of The Geysers, CA        35
6 km WNW of Cobb, CA              24
64 km ENE of Pedro Bay, Alaska    23
26 km SW of Jal, New Mexico       22
7 km WNW of Cobb, CA              21
63 km ENE of Pedro Bay, Alaska    20
9 km WNW of The Geysers, CA       20
65 km ENE of Pedro Bay, Alaska    17
Name: count, dtype: int64

In [53]:
df[df["mag"] >= 5.0][["time_utc", "place", "mag"]].sort_values("mag", ascending=False).head(10)


,time_utc,place,mag
3363,2026-01-02 13:58:18.346000+00:00,"4 km NNW of Rancho Viejo, Mexico",6.5
2027,2026-01-07 03:02:56.040000+00:00,"16 km ESE of Baculin, Philippines",6.4
1136,2026-01-10 14:58:23.481000+00:00,"245 km NNW of Tobelo, Indonesia",6.4
502,2026-01-13 07:34:08.725000+00:00,"133 km SE of Kuril’sk, Russia",6.2
3818,2026-01-01 01:53:01.814000+00:00,southeast Indian Ridge,6.0
4007,2025-12-31 14:26:56.732000+00:00,"91 km E of Noda, Japan",6.0
3137,2026-01-03 10:07:56.003000+00:00,southeast Indian Ridge,5.9
3765,2026-01-01 06:46:55.131000+00:00,"105 km NNW of Yakutat, Alaska",5.7
2364,2026-01-06 01:18:48.664000+00:00,"18 km S of Matsue, Japan",5.7
2786,2026-01-04 13:07:06.267000+00:00,Prince Edward Islands region,5.7


In [ ]:
from openai import OpenAI
client = OpenAI()




In [3]:
from openai import OpenAI
client = OpenAI()
print("OpenAI import OK")



OpenAI import OK


In [4]:
import pandas as pd 

df = pd.read_csv("../data/processed/earthquakes_processed.csv")
df.shape

(4158, 10)

In [5]:
summary = {
    "rows": int(df.shape[0]),
    "date_min": str(pd.to_datetime(df["time_utc"], utc=True).min()),
    "date_max": str(pd.to_datetime(df["time_utc"], utc=True).max()),
    "mag_min": float(df["mag"].min()),
    "mag_max": float(df["mag"].max()),
    "mag_mean": float(df["mag"].mean()),
    "count_mag_ge_5": int((df["mag"] >= 5.0).sum()),
    "top_places": df["place"].value_counts().head(10).to_dict(),
}

summary


{'rows': 4158,
 'date_min': '2025-12-31 00:03:35.983000+00:00',
 'date_max': '2026-01-14 23:49:34.240000+00:00',
 'mag_min': -1.44,
 'mag_max': 6.5,
 'mag_mean': 1.6446576013663443,
 'count_mag_ge_5': 76,
 'top_places': {'8 km NW of The Geysers, CA': 64,
  '7 km NW of The Geysers, CA': 45,
  '6 km NW of The Geysers, CA': 35,
  '6 km WNW of Cobb, CA': 24,
  '64 km ENE of Pedro Bay, Alaska': 23,
  '26 km SW of Jal, New Mexico': 22,
  '7 km WNW of Cobb, CA': 21,
  '63 km ENE of Pedro Bay, Alaska': 20,
  '9 km WNW of The Geysers, CA': 20,
  '65 km ENE of Pedro Bay, Alaska': 17}}

In [ ]:
## 6. AI-assisted Insights

In [7]:
from openai import OpenAI

client = OpenAI()

prompt = f"""
You are a data analyst.

Based on the following earthquake dataset summary:
- Write 5 clear analytical insights
- Mention any notable geographic patterns
- Highlight the distribution of strong earthquakes (magnitude >= 5)

Dataset summary:
{summary}
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

print(response.output_text)


Here are the analytical insights based on the earthquake dataset summary:

1. **Dataset Size and Timeframe**  
   The dataset comprises 4,158 earthquake records spanning approximately two weeks, from December 31, 2025, to January 14, 2026. This provides a short-term but dense snapshot of seismic activity.

2. **Magnitude Range and Average Intensity**  
   Earthquake magnitudes range from -1.44 (likely indicating very minor or incorrectly recorded events) up to 6.5, with a mean magnitude of about 1.64. This suggests most events are minor or light tremors.

3. **Strong Earthquake Distribution (Magnitude ≥ 5)**  
   There are 76 earthquakes with a magnitude of 5 or higher, representing roughly 1.8% of the total events. Though infrequent, these stronger earthquakes are significant due to their potential impact.

4. **Geographic Concentration of Events**  
   The highest concentrations of earthquakes occur near The Geysers area in California—the top three locations near “NW of The Geysers, 

In [ ]:
## 5. Event Severity Classification

Events are classified into Low, Medium, and High impact based on magnitude thresholds.

In [8]:
def classify_earthquake(mag: float) -> str:
    if mag < 4.0:
        return "Low" 
    elif mag < 5.5:
        return "Medium" 
    else:
        return "High" 
    

In [9]:
df["impact_level"] = df["mag"].apply(classify_earthquake)
df["impact_level"].value_counts()

impact_level
Low       3841
Medium     296
High        21
Name: count, dtype: int64

In [11]:
df[df["impact_level"] == "High"][["time_utc", "place", "mag"]].sort_values("mag", ascending=False).head(10)


,time_utc,place,mag
3363,2026-01-02 13:58:18.346000+00:00,"4 km NNW of Rancho Viejo, Mexico",6.5
2027,2026-01-07 03:02:56.040000+00:00,"16 km ESE of Baculin, Philippines",6.4
1136,2026-01-10 14:58:23.481000+00:00,"245 km NNW of Tobelo, Indonesia",6.4
502,2026-01-13 07:34:08.725000+00:00,"133 km SE of Kuril’sk, Russia",6.2
3818,2026-01-01 01:53:01.814000+00:00,southeast Indian Ridge,6.0
4007,2025-12-31 14:26:56.732000+00:00,"91 km E of Noda, Japan",6.0
3137,2026-01-03 10:07:56.003000+00:00,southeast Indian Ridge,5.9
2786,2026-01-04 13:07:06.267000+00:00,Prince Edward Islands region,5.7
3765,2026-01-01 06:46:55.131000+00:00,"105 km NNW of Yakutat, Alaska",5.7
2364,2026-01-06 01:18:48.664000+00:00,"18 km S of Matsue, Japan",5.7


In [ ]:
#“I implemented a rule-based impact classification to categorize seismic events into low, medium, and high impact levels, making the dataset more interpretable for non-technical stakeholders.”

In [14]:
df[df["impact_level"] == "High"]["mag"].describe()

count    21.000000
mean      5.785714
std       0.333595
min       5.500000
25%       5.500000
50%       5.600000
75%       6.000000
max       6.500000
Name: mag, dtype: float64

In [15]:
df.to_csv("../data/processed/earthquakes_processed_v2.csv", index=False)
